# Screenase walkthrough — plan → simulate → analyze

This notebook runs the full Screenase loop end-to-end, no bench required. Cells are small so GitHub renders them inline.

1. Load the example config
2. Build the design
3. Compute pipetting volumes + validate
4. Render the bench sheet (show the first 2 KB)
5. Simulate yields from a known effect structure
6. Fit the OLS model and rank effects
7. Draw the Pareto plot

In [ ]:
import numpy as np
import pandas as pd

from screenase.analyze import fit_model, pareto_plot, rank_effects
from screenase.bench_sheet import build_context, render_bench_sheet
from screenase.config import config_hash, load_config
from screenase.design import build_design
from screenase.volumes import compute_volumes, validate_volumes

cfg = load_config("../examples/config.yaml")
cfg

In [ ]:
design = build_design(cfg)
design.head()

In [ ]:
vol_df = compute_volumes(design, cfg)
warnings = validate_volumes(vol_df, cfg)
print(f"{len(warnings)} warning(s): {[w.reagent for w in warnings[:3]]}...")
vol_df.filter(like="_pipet_uL").head()

In [ ]:
ctx = build_context(
    vol_df, design["is_center"], cfg,
    run_id="walkthrough",
    generated_at="2026-04-16",
    lib_version="0.1.0",
    config_hash=config_hash(cfg),
    warnings=warnings,
)
html = render_bench_sheet(ctx)
print(html[:2000])

In [ ]:
# Simulate yields: y = 3 x_NTPs - 2 x_MgCl2 + 1.5 x_NTPs * x_MgCl2 + small noise
rng = np.random.default_rng(0)
x1 = design["NTPs_mM_each_coded"].to_numpy()
x2 = design["MgCl2_mM_coded"].to_numpy()
yields = 3.0 * x1 - 2.0 * x2 + 1.5 * x1 * x2 + rng.normal(0, 0.1, size=len(design))
results = design.copy()
results["yield_ug_per_uL"] = yields
results[["NTPs_mM_each", "MgCl2_mM", "yield_ug_per_uL"]].head()

In [ ]:
factor_cols = [c for c in results.columns if c.endswith("_coded")]
fit = fit_model(results, "yield_ug_per_uL", factor_cols)
effects = rank_effects(fit)
pd.DataFrame([(e.term, e.coef, e.t, e.p) for e in effects],
             columns=["term", "coef", "t", "p"])

In [ ]:
from pathlib import Path
from IPython.display import Image

out = Path("/tmp/walkthrough_pareto.png")
pareto_plot(effects, out, df_resid=int(fit.df_resid))
Image(str(out))

The top three effects — `NTPs_mM_each_coded`, `MgCl2_mM_coded`, and their interaction — match the ground-truth structure, and every negligible term sits well left of the significance line. That's the whole loop.

From here: save `results` as a CSV and the same analysis runs from the CLI (`screenase analyze ...`) or the Streamlit app's Analyze tab.